# Медицина.
## Анализ факторов развития болезни

Коробовцева Ольга  
Лоза Александр  
Павлова Анна  

## Установка и импорт необходимых библиотек


В проекте используются: numpy, pandas, matplotlib, seaborn, sklearn, catboost

In [19]:
# Медицина.
## Анализ факторов развития болезни

Коробовцева Ольга  
Лоза Александр  
Павлова Анна  

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.1 -> 23.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Установка и импорт необходимых библиотек


В проекте используются: numpy, pandas, matplotlib, seaborn, sklearn, catboost

In [ ]:
%pip install pandas numpy matplotlib seaborn scipy scikit-learn

In [ ]:
import pandas as pd                     
import numpy as np                      
import matplotlib.pyplot as plt         
import seaborn as sns                   
import scipy.stats as stats             
import regression
import matplotlib
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.dummy import DummyRegressor

ModuleNotFoundError: No module named 'regression'

In [ ]:
%matplotlib inline


## Предобработка данных

Дополнительные факторы

In [ ]:
def add_column(df: pd.DataFrame) -> None:
    df["аик+переливание_крови"] = df["аик"] * (df["объем_гемотрансфузии"].apply(lambda x: 1 if x > 0 else 0))

Удаление дубликатов

In [ ]:
def check_duplicates(df: pd.DataFrame) -> None:
    df.drop_duplicates(inplace=True)

Удаление строк, содержащих пропуски

In [ ]:
def drop_nan(df: pd.DataFrame) -> None:
    df.dropna(inplace=True)

Исправление неправильных символов в датасете, кодирование категориальных переменных


In [ ]:
def fix_types(df: pd.DataFrame) -> None:
    df["развитие_опп"] = df["развитие_опп"].apply(lambda x: 0 if x == "нет" else 1)
    df["хбп"] = df["хбп"].apply(lambda x: 0 if x == "Пациенты без ХБП" else 1 if x == "Стадия C1-C2" else 2)
    for column in df.select_dtypes(include=["object"]).columns:
        df[column] = df[column].apply(lambda x: float(x.replace(",", ".").replace("o", "0")) if type(x) != float else x)

Поиск и удаление выбросов

In [ ]:
def drop_outliers(df: pd.DataFrame, a=3) -> tuple:
    columns = df.columns
    outliers_all = np.array([False for i in range(len(df))])
    for column in columns:
        if len(df[column].unique()) > len(df) * 0.05:
            std = df[column].std()
            median = df[column].median()
            outliers = abs(df[[column]] - median) > a * std
            outliers_all = np.bitwise_or(outliers.to_numpy().flatten(), outliers_all)
    df_unchanged = df.copy()
    df = df[~ outliers_all].dropna()
    return df, df_unchanged, outliers_all

Обработика исходных данных

In [ ]:
def preprocess(df_name: str) -> pd.DataFrame:
    df = pd.read_csv(df_name)
    df.columns = [x.lower().replace(" ", "_").replace(",", "") for x in df.columns]
    df.columns = [x[:-1] if x[-1] == "_" else x for x in df.columns]
    fix_types(df)
    drop_nan(df)
    check_duplicates(df)
    df, unchanged_df, outliers = drop_outliers(df)
    add_column(df)
    return df.reset_index()

In [ ]:
df = preprocess("medics_1.csv")
df

## Общий анализ

Функция для проверки, является ли столбец категориальным

In [ ]:
def is_categorical(df: pd.DataFrame, column: str) -> bool:
    return len(df[column].unique()) < len(df) * 0.05


Подсчёт корреляции между факторами и целевой переменной

In [ ]:
def check_correlations(df: pd.DataFrame) -> list:
    columns = ["возраст", "сахарный_диабет", "гб", "хбп", "сад", "дад", "чсс", "рн", "фракция_изгнания", "холестерин",
               "креатинин_крови", "мочевина", "скф_расч.", "калий", "имт", "толщина_паренхимы_почек"]
    res = []
    for x in columns:
        if is_categorical(df, x):
            temp = stats.chi2_contingency(pd.crosstab(df[x], df["развитие_опп"]))
            res.append([x, "развитие_опп", round(temp[1], 4), "Хи-квадрат"])
        else:
            if stats.shapiro(df[x])[1] > 0.05:
                temp = stats.ttest_ind(df["развитие_опп"], df[x])
                res.append([x, "развитие_опп", round(temp[1], 4), "T-критерий Стьюента"])
            else:
                temp = stats.mannwhitneyu(df["развитие_опп"], df[x])
                res.append([x, "развитие_опп",round(temp[1], 4), "U-критерий Манна-Уитни"])    
    return pd.DataFrame(np.array(res), columns=["Фактор", "Развитие ОПП", "p-уровень", "Метод"])

Функция для получения стадии ХБП по СКФ

In [ ]:
def get_diagnosis(m: int) -> str:
    if m > 100:
        return "Пациенты без ХБП"
    if m > 60:
        return "Стадия C1-C2"
    return "Стадия С3"


Проверка стадии ХБП по СКФ

In [ ]:
def check_diagnosis(df: pd.DataFrame) -> list:  # ?
    num_to_words = {0: "Пациенты без ХБП", 1: "Стадия C1-C2", 2: "Стадия С3"}
    wrong_diagnosis = []
    for idx, row in df.iterrows():
        if row["скф_расч."] > 100:
            if row["хбп"] != 0:
                print(
                    f"У пациента {idx} диагноз {num_to_words[row['хбп']]}, но должен быть {get_diagnosis(row['скф_расч.'])}")
                wrong_diagnosis.append(idx)
        elif row["скф_расч."] > 60:
            if row["хбп"] != 1:
                print(
                    f"У пациента {idx} диагноз {num_to_words[row['хбп']]}, но должен быть {get_diagnosis(row['скф_расч.'])}")
                wrong_diagnosis.append(idx)
        elif row["скф_расч."] != 2:
            print(
                f"У пациента {idx} диагноз {num_to_words[row['хбп']]}, но должен быть {get_diagnosis(row['скф_расч.'])}")
            wrong_diagnosis.append(idx)
    return wrong_diagnosis

In [ ]:
def poch_paren(df: pd.DataFrame) -> None:
    poch_paren = df[df["хбп"] == 0]
    print(stats.pearsonr(poch_paren["толщина_паренхимы_почек"], poch_paren["возраст"]))
    sns.scatterplot(data=poch_paren, x="толщина_паренхимы_почек", y="возраст")
    plt.show()

In [ ]:
def imt_hol(df: pd.DataFrame) -> None:
    # !!! норма - 18,5-25 избыточный - 25-30, больше - ожирение
    # норма 3,6-7,8 ммоль и моль!!!
    df["ктг_имт"] = df["имт"].apply(lambda x: 1 if x > 25 else 0)
    df["ктг_холестерин"] = df["холестерин"].apply(lambda x: 1 if x > 7.8 else 0)
    imt_hol = pd.crosstab(df['ктг_имт'], df['ктг_холестерин'])
    print(imt_hol)
    print(stats.chi2_contingency(pd.crosstab(df['ктг_имт'], df['ктг_холестерин'])))
    sns.heatmap(imt_hol, cmap="YlGnBu", annot=True, cbar=False);
    plt.show()

In [ ]:
def dependence_time_fact(df: pd.DataFrame) -> None:
    print(stats.pointbiserialr(df["инфаркт_миокарда"], df["длительность_операции"]))
    infarct_0 = df[df["инфаркт_миокарда"] == 0]
    infarct_1 = df[df["инфаркт_миокарда"] == 1]
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    plot = sns.boxplot(ax=axes[0], data=infarct_0, y="длительность_операции")
    plot2 = sns.boxplot(ax=axes[1], data=infarct_1, y="длительность_операции")
    plt.show()

Подсчет комбинаций *"хроническая болезнь + развитие опп"*

In [ ]:
def count_disease(df: pd.DataFrame, imt: str) -> list:
    lst = ["гб", "стенокардия", "инфаркт_миокарда", "желудочковая_экстрасистолия",
           "мерцательная_аритмия", "хсн", "нк"]
    answer = []
    for i in range(len(lst)):
        answer.append(df.loc[(df['имт_ном'] == imt) & (df[lst[i]] == 1), 'развитие_опп'].sum())
    return answer

Графическое отображение вышеописанной функции с использованием *pie chart*

In [ ]:
def draw_chart(labels, sizes, name, fig, ax, row, column):
        colors = ['#506D2F', '#2a2922', '#f3ebdd', '#7d5642', "#626D71", "#cdcdc0", "#DDBC95"]

        wedges, texts = ax[row][column].pie(sizes, startangle=-40, colors=colors)

        centre_circle = plt.Circle((0, 0), 0.70, fc='white')
        fig = plt.gcf()
        fig.gca().add_artist(centre_circle)

        ax[row][column].legend(wedges, labels,
                title="болезни",
                loc="center left",
                bbox_to_anchor=(1, 0, 0.5, 1))

        plt.setp(texts, size=12, weight="bold")
        ax[row][column].pie(sizes, labels=labels, colors=colors,
                autopct='%1.1f%%')
        ax[row][column].set_title(name)


    
Введение нового фактора **"имт_ном"**. Функция считывает значение "имт" из датасета, обрабатывает и записывает соответствующее номинальное значение.

Выбор и классификация болезней, которые оказывают влияние на сердце:
* Болезни, ведущие к сердечным заболеваниям
    + Гипертония
    + Сахарный диабет
* Синдромы
    + Стенокардия
* Болезни сердца
    + Инфаркт миокарда
    + Желудочковая экстрасистолия
    + Мерцательная аритмия
    + Хроническая сердечная недостаточность
    + Недостаточность кровообращения
    
Отосительно перечисленных болезней были найдены зависимости относительно нового каждой группы из **"имт_ном"**
    

In [ ]:
def imt(df):

    # Подпункт 2
    df["имт_ном"] = df["имт"].apply(lambda x: "выраженный_дефицит_массы_тела" if x < 16
                                    else "недостаточная_масса_тела" if 16 <= x < 18.5
                                    else "норма" if 18.5 <= x < 25
                                    else "избыточная_масса_тела" if 25 <= x < 30
                                    else "ожирение_1_степени" if 30 <= x < 35
                                    else "ожирение_2_степени" if 35 <= x < 40
                                    else "ожирение_3_степени")
    
    labels = ["ГБ", "Стенокардия", "Инфаркт миокарда", "Желудочковая экстрасистолия", "Мерцательная аритмия",
              "ХСН", "НК"]

    sizes_norma = count_disease(df, "норма")
    sizes_overweight = count_disease(df, "избыточная_масса_тела")
    sizes_one = count_disease(df, "ожирение_1_степени")
    sizes_two = count_disease(df, "ожирение_2_степени")
    fig, ax = plt.subplots(2, 2, figsize=(36, 18))

    draw_chart(labels, sizes_norma, "норма", fig, ax, 0, 0)
    draw_chart(labels, sizes_overweight, "избыточная_масса_тела", fig, ax, 0, 1)
    draw_chart(labels, sizes_one, "ожирение_1_степени", fig, ax, 1, 0)
    draw_chart(labels, sizes_two, "ожирение_2_степени", fig, ax, 1, 1)
    plt.show()

Подсчет процент пациентов с хроническими заболеваниям:
* Сахарный диабет
* Гипертония
* Хроническая болезнь почек

Для групп: *"есть ОПП", "нет ОПП"*

In [23]:
def chronic_diseases(df: pd.DataFrame) -> None:
    # Подпункт 1
    
    diabetes_mellitus_true = round(((df.loc[df['развитие_опп'] == 1, 'сахарный_диабет']
                                     .sum()) / (sum(df["развитие_опп"]))) * 100, 4)
    diabetes_mellitus_false = round(((df.loc[df['развитие_опп'] == 0, 'сахарный_диабет']
                                      .sum()) / (len(df['развитие_опп']) - sum(df["развитие_опп"]))) * 100, 4)

    hypertension_true = round(((df.loc[df['развитие_опп'] == 1, 'гб']
                                .sum()) / (sum(df["развитие_опп"]))) * 100, 4)
    hypertension_false = round(((df.loc[df['развитие_опп'] == 0, 'гб']
                                 .sum()) / (len(df['развитие_опп']) - sum(df["развитие_опп"]))) * 100, 4)

    df["хбп_бин"] = df["хбп"].apply(lambda x: 1 if x == 1 or x == 2 else 0)
    chronic_kidney_disease_true = round(((df.loc[df['хбп_бин'] == 1, 'развитие_опп']
                                          .sum()) / (sum(df["развитие_опп"]))) * 100, 4)
    chronic_kidney_disease_false = round(((df.loc[df['развитие_опп'] == 0, 'хбп_бин']
                                           .sum()) / (len(df['развитие_опп']) - sum(df["развитие_опп"]))) * 100, 4)

    print(str(diabetes_mellitus_true) + "%", str(diabetes_mellitus_false) + "%")
    print(str(hypertension_true) + "%", str(hypertension_false) + "%")
    print(str(chronic_kidney_disease_true) + "%", str(chronic_kidney_disease_false) + "%")
 

### Визуализация результатов анализа

In [ ]:
imt(df)

In [ ]:
wrong_diagnsis = check_diagnosis(df)

In [ ]:
plt.pie((len(df) - len(wrong_diagnsis), len(wrong_diagnsis)), colors=["#506D2F", "#626D71"], autopct='%1.1f%%')
plt.legend(["Диагнозх поставлен верно", "Диагноз поставлен неверно"], loc="lower right")
plt.show()

Подсчёт p-уровня корреляции между различными факторами

In [ ]:
check_correlations(df)

In [ ]:
correlations = check_correlations(df).to_numpy().T
sns.scatterplot(x=correlations[2], y=correlations[0])

In [ ]:
dependence_time_fact(df)


In [ ]:
dependence_time_fact(df)


In [ ]:
imt_hol(df)


In [ ]:
poch_paren(df)

In [ ]:
def is_normal(df, col_2):
    if stats.shapiro(df[col_2])[1] >= 0.05:
        c_p =  stats.ttest_ind(df['развитие_опп'], df[col_2])

    else:
        c_p =  stats.ttest_ind(df['развитие_опп'], df[col_2])
    print(round(c_p[0], 4), round(c_p[1], 4))
    sns.boxplot(data=df, x='развитие_опп', y=col_2)
    plt.show()

In [ ]:
def chi_2(df, col_2):
    cross_tab = pd.crosstab(df['развитие_опп'], df[col_2])
    c_p =  stats.chi2_contingency(cross_tab)
    print(round(c_p[0], 4), round(c_p[1], 4))
    sns.heatmap(cross_tab, cmap="YlGnBu", annot=True, cbar=False);
    plt.show()

In [ ]:
def hypo_1(df) -> None:  # 'хбп' хи квадрат
    chi_2(df, 'хбп')

In [ ]:
def hypo_2(df) -> None:  # 'возраст' статы маняуитни
    is_normal(df, "возраст")
    print( stats.pointbiserialr(df['развитие_опп'], df["возраст"]))


In [ ]:

def hypo_3(df) -> None:  # 'пол'
    chi_2(df, 'пол')



In [ ]:
def hypo_4(df) -> None:  # 'мочевина' маняуитни
    is_normal(df, 'мочевина')
    print( stats.pointbiserialr(df['развитие_опп'], df["мочевина"]))


In [ ]:
def hypo_5(df) -> None:  # 'лейкоциты_крови' маняуитни
    is_normal(df, 'лейкоциты_крови')
    print( stats.pointbiserialr(df['развитие_опп'], df["лейкоциты_крови"]))

In [ ]:
def hypo_6(df) -> None:  # 'тромбоциты'маняуитни
    is_normal(df, "тромбоциты")
    print( stats.pointbiserialr(df['развитие_опп'], df["тромбоциты"]))


In [ ]:
def hypo_7(df) -> None:  # 'общий_белок'маняуитни
    is_normal(df, "общий_белок")
    print( stats.pointbiserialr(df['развитие_опп'], df["общий_белок"]))


In [ ]:
def hypo_8(df) -> None:  # 'аик'
    chi_2(df, 'аик')


In [ ]:
def hypo_9(df) -> None:  # 'объем_гемотрансфузии'маняуитни
    is_normal(df, "объем_гемотрансфузии")
    print( stats.pointbiserialr(df['развитие_опп'], df["объем_гемотрансфузии"]))

In [ ]:
def hypo_10(df):  # 'диурез'маняуитни
    is_normal(df, "диурез")
    print( stats.pointbiserialr(df['развитие_опп'], df["диурез"]))

In [ ]:
hypo_1(df)

In [ ]:
hypo_2(df)

In [ ]:
hypo_3(df)

In [ ]:
hypo_4(df)

In [ ]:
hypo_5(df)

In [ ]:
hypo_6(df)


In [ ]:
hypo_7(df)


In [ ]:
hypo_8(df)


In [ ]:
hypo_9(df)


In [ ]:
hypo_10(df)


## Регрессионная модель

Обучение модели, вывод RMSE и сравнение с DummyRegressor из Sklearn

In [ ]:
def fit_model(X: np.array, y: np.array, n_estimators=10000, max_depth=3, random_seed=42, plot=False) -> CatBoostRegressor:
    regressor = CatBoostRegressor(n_estimators=n_estimators, max_depth=max_depth, learning_rate=0.001, random_seed=random_seed)
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8)
    regressor.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True, plot=plot, plot_file="train.html")
    print(f"RMSE при  определении СКФ: {mean_squared_error(regressor.predict(X_test), y_test) ** 0.5}")
    dummy = DummyRegressor()
    dummy.fit(X_train, y_train)
    print(f"RMSE при  определении СКФ с помощью DummyRegressor: {mean_squared_error(dummy.predict(X_test), y_test) ** 0.5}")
    regressor.save_model("regressor.cbm")
    print("Модель сохранена как regressor.cbm")
    return regressor


Функция для предсказания значения СКФ

In [ ]:
def predict(X: np.array, model="regressor.cbm") -> float:
    regressor = CatBoostRegressor()
    regressor.load_model(model)
    return regressor.predict(X)


Приемр обучения модели

In [ ]:
model = fit_model(df[['возраст', 'пол', 'гб', 'сахарный_диабет',
'стенокардия', 'инфаркт_миокарда', 'мерцательная_аритмия',
'желудочковая_экстрасистолия', 'а-в_блокада',
'блокада_ножек_пучка_гиса', 'сад', 'дад', 'креатинин_крови',
'мочевина', 'калий', 'натрий', 'хлориды', 'кальций', 'рн',
'ве', 'нсо3', 'ро2', 'рсо2', 'оксигем.', 'общ.со2', 'гемоглобин',
'лейкоциты_крови', 'тромбоциты', 'холестерин',
'триглицериды', 'лпонп', 'лпнп', 'общий_белок', 'имт']], df[['скф_расч.']], plot=True)